# T60 AUV 最新精度 Policy 可视化

只评估当前 `t60_trajectory_precision_v11.json` 对应的 `t60_precision_v11` 训练运行，并从中选择最新的非零 checkpoint。该 policy 以 25 Hz 更新，使用无注入噪声且确定性延迟 50 ms 的融合状态、`precision_v6` 全 rad 奖励、201 维 `mlp_history_8` Actor、RSL-RL 原生 adaptive KL、完整 6×6 CFD 水动力，以及最终达到质量 ±10%、质心逐轴 ±5 cm、质心到浮心偏置逐分量 ±10%、逐台推进器增益 ±15% 和全体推进器共同弱化 0–15% 的 DR v7 课程。目标姿态始终保持 `roll=pitch=0`，仅让 yaw 跟随水平速度方向；训练仅使用三轴纯正弦与横向/垂向前进正弦，Lissajous 在这里作为未参与训练的组合泛化评估。

Isaac Sim 中的名义评估轨迹为包络 **5 × 3 × 1 m**、目标线速度 **0.14 m/s** 的三维 Lissajous；该速度在当前加速度、jerk 与姿态角速度约束下不触发重定时。评估始终启用安全边界，并分别报告全程与稳态误差。

> 这是开阔水域虚拟安全盒评估；1 m 垂向包络不能原样放进 0.75 m 深的实物水池。

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
from IPython.display import display
from simulation.domain_randomization import load_domain_randomization_spec_json
from simulation.training.campaign import configure_plots, run_command
from simulation.training.evaluation.metrics import (
    collect_summary_df, load_eval_log, plot_eval_detail, quick_numeric_report,
)
from simulation.training.recipe import (
    ExperimentSpec, load_training_recipe, run_input_paths,
)

configure_plots()
ISAACLAB_ROOT = Path.home() / 'IsaacLab'
RLPOLICY_ROOT = REPO_ROOT / 'simulation/rlpolicy'
RECIPE_PATH = REPO_ROOT / 'simulation/training/recipes/t60_trajectory_precision_v11.json'
RUN_NAME = 't60_precision_v11'
RECIPE = load_training_recipe(RECIPE_PATH)
SPEC = ExperimentSpec(
    isaaclab_root=ISAACLAB_ROOT,
    rlpolicy_root=RLPOLICY_ROOT,
    mlp_architecture=RECIPE.mlp_architecture,
)
CHECKPOINT_PATH = max(
    (checkpoint_path for checkpoint_path in SPEC.logs_root.glob(f'*_{RUN_NAME}/model_*.pt')
     if int(checkpoint_path.stem.removeprefix('model_')) > 0),
    key=lambda checkpoint_path: checkpoint_path.stat().st_mtime,
)
POLICY_RUN_DIR = CHECKPOINT_PATH.parent
CHECKPOINT = CHECKPOINT_PATH.name
ACTIVE_RUN = POLICY_RUN_DIR.name
RUN_INPUTS = run_input_paths(POLICY_RUN_DIR)
RUN_RECIPE = load_training_recipe(RUN_INPUTS.recipe)
RUN_DR = load_domain_randomization_spec_json(RUN_INPUTS.domain_randomization)

TRAJECTORY = 'lissajous'
TRAJECTORY_MAX_SPEED_MPS = 0.14
TRAJECTORY_SIZE_X_M = 5.0
TRAJECTORY_SIZE_Y_M = 3.0
TRAJECTORY_SIZE_Z_M = 1.0
TRAJECTORY_AMP_X_M = TRAJECTORY_SIZE_X_M / 2.0
TRAJECTORY_AMP_Y_M = TRAJECTORY_SIZE_Y_M / 2.0
TRAJECTORY_AMP_Z_M = TRAJECTORY_SIZE_Z_M / 2.0
DURATION_S = 140.0
EVALUATION_LABEL = (
    f'{Path(CHECKPOINT).stem}_5x3x1_lissajous_vmax_{TRAJECTORY_MAX_SPEED_MPS:.2f}'
    .replace('.', 'p')
)

print(f'run: {POLICY_RUN_DIR}')
print(f'checkpoint: {CHECKPOINT}')
print(f'recipe: {RUN_RECIPE.name} / reward: {RUN_RECIPE.reward_profile}')
print(f'DR: {RUN_DR.name} / schema: {RUN_DR.schema_version}')
print(f'trajectory: {TRAJECTORY} with max speed {TRAJECTORY_MAX_SPEED_MPS:.2f} m/s')
print(f'curve envelope: {TRAJECTORY_SIZE_X_M:.1f} × {TRAJECTORY_SIZE_Y_M:.1f} × {TRAJECTORY_SIZE_Z_M:.1f} m')


## 启动 Isaac Sim 实时可视化

蓝线为目标，橙线为真实轨迹。140 秒结束后窗口自动关闭。

In [ ]:
VISUALIZATION_COMMAND = [
    './isaaclab.sh', '-p', SPEC.eval_script,
    '--task', SPEC.task_name,
    '--checkpoint', str(CHECKPOINT_PATH),
    '--trajectory', TRAJECTORY,
    '--trajectory_speed', str(TRAJECTORY_MAX_SPEED_MPS),
    '--trajectory_amp_x', str(TRAJECTORY_AMP_X_M),
    '--trajectory_amp_y', str(TRAJECTORY_AMP_Y_M),
    '--trajectory_amp_z', str(TRAJECTORY_AMP_Z_M),
    '--duration', str(DURATION_S),
    '--num_envs', '1',
    '--seed', '42',
    '--evaluation_label', EVALUATION_LABEL,
    '--align_initial_target',
]
run_command(
    VISUALIZATION_COMMAND,
    cwd=ISAACLAB_ROOT,
    execute=True,
    label=f'{Path(CHECKPOINT).stem.upper()} LIVE VISUALIZATION',
)


## 跟踪结果

In [ ]:
summary_df = collect_summary_df(SPEC, ACTIVE_RUN, case_label=EVALUATION_LABEL)
summary_df = summary_df[
    (summary_df['checkpoint_name'] == CHECKPOINT)
    & (summary_df['trajectory'] == TRAJECTORY)
].copy()
display(summary_df[[
    'checkpoint_name', 'position_rmse', 'steady_position_rmse',
    'steady_position_error_p95', 'steady_position_bias_norm_m',
    'steady_position_x_rmse_m', 'steady_position_y_rmse_m',
    'steady_position_z_rmse_m', 'steady_cross_track_position_error_rmse_m',
    'steady_attitude_error_rmse_deg', 'steady_mean_nose_to_target_heading_angle_deg',
    'mean_thruster_wrench_force_norm_n', 'minimum_vehicle_boundary_clearance_m',
    'mean_reward_per_step',
]])
display(quick_numeric_report(summary_df))

detail_df, detail_path = load_eval_log(
    SPEC, ACTIVE_RUN, CHECKPOINT, TRAJECTORY, EVALUATION_LABEL,
)
print(f'loaded {len(detail_df)} time-aligned rows from {detail_path}')
detail_figure = plot_eval_detail(
    SPEC, ACTIVE_RUN, detail_df, CHECKPOINT, TRAJECTORY,
    case_label=EVALUATION_LABEL, env_id=0, save=True,
)
display(detail_figure)
plt.close(detail_figure)
